# BGGTDM v2 — New Feature Engineering

Adds three feature groups to `game_logs_enriched.csv`:
- **Feature A** — Opponent WR/TE TD rate allowed (rolling 4-game lag, from existing data)
- **Feature B** — Snap count % (lag + rolling, via `nfl_data_py`)
- **Feature C** — Red zone targets (rolling, rz_target_share, rz_td_rate, via `nfl_data_py` PBP)

Run this notebook once. Then run `train.py` (or notebook 04) to retrain with full feature set.

In [1]:
import os
os.chdir('/Users/aaronangeles/Documents/Projects/Active/biggamegabefallacy/ml')

import numpy as np
import pandas as pd

pd.options.display.max_columns = 60
pd.options.display.float_format = '{:.4f}'.format

df = pd.read_csv('data/game_logs_enriched.csv')
print(f'Loaded {len(df)} rows | seasons: {sorted(df["season"].unique())}')

# Drop any previously-added columns so notebook is idempotent on re-runs
cols_to_regen = [
    'opp', 'opp_wr_te_td_rate_allowed', 'opp_wr_te_targets_pg_allowed',
    'lag_snap_pct', 'roll3_snap_pct', 'season_snap_pct',
    'cum_rz_targets', 'cum_rz_tds', 'roll3_rz_targets', 'rz_target_share',
    'name_norm',
]
df.drop(columns=[c for c in cols_to_regen if c in df.columns], inplace=True)
print(f'Columns after dropping previously-added features ({len(df.columns)}): {list(df.columns)}')

Loaded 9972 rows | seasons: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Columns after dropping previously-added features (57): ['player_id', 'name', 'pos', 'team', 'game_id', 'season', 'week', 'receptions', 'targets', 'rec_yards', 'rec_tds', 'long_rec', 'rec_avg', 'scored_td', 'target_bucket', 'lag_yards', 'lag_targets', 'lag_receptions', 'cum_yards', 'cum_targets', 'cum_receptions', 'cum_tds', 'games_played', 'yards_pg', 'targets_pg', 'receptions_pg', 'roll3_yards', 'roll3_targets', 'roll3_receptions', 'td_rate_per_target', 'lag_ypr', 'target_trend', 'weeks_since_td', 'is_te', 'is_week1', 'exp', 'catch_rate_raw', 'lag_catch_rate', 'roll3_catch_rate', 'season_catch_rate', 'lag_ypt', 'season_ypt', 'roll3_long_rec', 'cr_q', 'roll3_target_std', 'target_cv', 'season_target_std', 'td_streak', 'td_last1', 'td_last2', 'td_last3', 'tds_last3', 'height_in', 'weight', 'exp_tier', 'height_bucket', 'is_home']


## Feature A — Opponent WR/TE TD Rate Allowed

Derivable from existing data — no new fetch needed.

**Approach:** For each row, parse the opponent from `game_id` (`YYYYMMDD_AWAY@HOME`).  
Aggregate total WR/TE TDs and targets *scored against* each defense per game.  
Use a rolling 4-game lag (shifted by 1) to avoid leakage.

**New columns:** `opp_wr_te_td_rate_allowed`, `opp_wr_te_targets_pg_allowed`

In [2]:
# --- Parse opponent from game_id ---
# game_id format: 20221020_NO@ARI  (YYYYMMDD_AWAY@HOME)
def get_opponent(row):
    away, home = row['game_id'].split('_')[1].split('@')
    return home if row['team'] == away else away

df['opp'] = df.apply(get_opponent, axis=1)

# --- Per-game offensive totals scored against each defense ---
# Group by the defensive team (opp) + game
def_agg = (
    df.groupby(['game_id', 'opp', 'season', 'week'])
    .agg(wr_te_tds_vs=('scored_td', 'sum'),
         wr_te_targets_vs=('targets', 'sum'))
    .reset_index()
    .rename(columns={'opp': 'def_team'})
)

# Sort chronologically per defense
def_agg = def_agg.sort_values(['def_team', 'season', 'week']).reset_index(drop=True)

# Rolling 4-game window, shifted by 1 (no leakage)
def roll4_lag(s):
    return s.shift(1).rolling(4, min_periods=1).sum()

def_agg['roll4_tds_allowed']    = def_agg.groupby('def_team')['wr_te_tds_vs'].transform(roll4_lag)
def_agg['roll4_targets_allowed'] = def_agg.groupby('def_team')['wr_te_targets_vs'].transform(roll4_lag)

def_agg['opp_wr_te_td_rate_allowed']   = (
    def_agg['roll4_tds_allowed'] / def_agg['roll4_targets_allowed'].replace(0, np.nan)
)
def_agg['opp_wr_te_targets_pg_allowed'] = def_agg['roll4_targets_allowed'] / 4

# Join back: player's opp = def_team, same game_id
df = df.merge(
    def_agg[['game_id', 'def_team', 'opp_wr_te_td_rate_allowed', 'opp_wr_te_targets_pg_allowed']],
    left_on=['game_id', 'opp'],
    right_on=['game_id', 'def_team'],
    how='left'
).drop(columns=['def_team'])

print(f'opp_wr_te_td_rate_allowed — null rate: {df["opp_wr_te_td_rate_allowed"].isna().mean():.1%}')
print(df['opp_wr_te_td_rate_allowed'].describe())

opp_wr_te_td_rate_allowed — null rate: 1.1%
count   9863.0000
mean       0.0443
std        0.0209
min        0.0000
25%        0.0294
50%        0.0430
75%        0.0566
max        0.2000
Name: opp_wr_te_td_rate_allowed, dtype: float64


In [3]:
# Inspect: top and bottom defenses by recent opp_wr_te_td_rate_allowed
season_2024 = df[df['season'] == 2024].copy()
def_profile = (
    season_2024.groupby('opp')[['opp_wr_te_td_rate_allowed', 'opp_wr_te_targets_pg_allowed']]
    .mean()
    .sort_values('opp_wr_te_td_rate_allowed', ascending=False)
)
print('Top 5 softest defenses (2024 avg):')
print(def_profile.head())
print('\nTop 5 toughest defenses (2024 avg):')
print(def_profile.tail())

Top 5 softest defenses (2024 avg):
     opp_wr_te_td_rate_allowed  opp_wr_te_targets_pg_allowed
opp                                                         
HOU                     0.0592                       21.9268
DAL                     0.0570                       20.2063
WSH                     0.0557                       21.0333
CLE                     0.0556                       21.9659
CAR                     0.0556                       22.2762

Top 5 toughest defenses (2024 avg):
     opp_wr_te_td_rate_allowed  opp_wr_te_targets_pg_allowed
opp                                                         
BAL                     0.0348                       24.1528
TB                      0.0332                       25.1994
DET                     0.0312                       27.3482
NYJ                     0.0304                       20.4778
NO                      0.0301                       24.8267


## Feature B — Snap Count %

Fetched via `nfl_data_py`. Free, no API key needed.

**New columns:** `lag_snap_pct`, `roll3_snap_pct`, `season_snap_pct`

In [4]:
try:
    import nfl_data_py as nfl
    print('nfl_data_py already installed')
except ImportError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-deps', 'nfl-data-py'],
        check=True
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', 'appdirs', 'pyarrow', 'requests', 'tqdm'],
        check=True
    )
    import nfl_data_py as nfl
    print('nfl_data_py installed')

print('Fetching snap counts 2022-2025...')
snaps_raw = nfl.import_snap_counts([2022, 2023, 2024, 2025])
print(f'Raw snap count rows: {len(snaps_raw)}')
print(f'Columns: {list(snaps_raw.columns)}')
snaps_raw.head(3)

nfl_data_py already installed
Fetching snap counts 2022-2025...


Raw snap count rows: 106148
Columns: ['game_id', 'pfr_game_id', 'season', 'game_type', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']


,game_id,pfr_game_id,season,game_type,week,player,pfr_player_id,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
0,2022_01_BAL_NYJ,202209110nyj,2022,REG,1,Max Mitchell,MitcMa02,T,NYJ,BAL,84.0000,1.0000,0.0000,0.0000,3.0000,0.1100
1,2022_01_BAL_NYJ,202209110nyj,2022,REG,1,Alijah Vera-Tucker,VeraAl00,G,NYJ,BAL,84.0000,1.0000,0.0000,0.0000,3.0000,0.1100
2,2022_01_BAL_NYJ,202209110nyj,2022,REG,1,George Fant,FantGe00,T,NYJ,BAL,84.0000,1.0000,0.0000,0.0000,3.0000,0.1100


In [5]:
# Filter to WR/TE, select relevant columns
snaps = (
    snaps_raw[snaps_raw['position'].isin(['WR', 'TE'])]
    [['player', 'season', 'week', 'team', 'offense_pct']]
    .rename(columns={'player': 'name', 'offense_pct': 'snap_pct'})
    .copy()
)
snaps['snap_pct'] = pd.to_numeric(snaps['snap_pct'], errors='coerce')
snaps['season']   = snaps['season'].astype(int)
snaps['week']     = snaps['week'].astype(int)

# nflverse uses different team abbreviations than Tank01 — normalize to Tank01 convention
TEAM_OVERRIDES = {'LA': 'LAR', 'WAS': 'WSH'}
snaps['team'] = snaps['team'].replace(TEAM_OVERRIDES)

# Name crosswalk: Tank01 names → nflverse format
NAME_OVERRIDES = {
    'kyle pitts sr.': 'kyle pitts',
    'dk metcalf': 'd.k. metcalf',
    'marvin mims jr.': 'marvin mims',
    'brian thomas jr.': 'brian thomas',
    'john metchie iii': 'john metchie',
    'chig okonkwo': 'chigoziem okonkwo',       # nflverse uses full first name
    'joshua palmer': 'josh palmer',             # Tank01 full vs nflverse short
    'hollywood brown': 'marquise brown',        # Tank01 nickname vs nflverse legal
    'chris godwin jr.': 'chris godwin',         # Strip Jr. suffix
    # add others as discovered
}

# Normalize name for fuzzy join (lowercase, strip), then apply overrides
df['name_norm']    = df['name'].str.lower().str.strip().replace(NAME_OVERRIDES)
snaps['name_norm'] = snaps['name'].str.lower().str.strip().replace(NAME_OVERRIDES)

# Compute rolling features within each player-team-season group
snaps = snaps.sort_values(['name_norm', 'team', 'season', 'week']).reset_index(drop=True)
g = snaps.groupby(['name_norm', 'team', 'season'])

snaps['lag_snap_pct']    = g['snap_pct'].shift(1)
snaps['roll3_snap_pct']  = g['snap_pct'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
snaps['season_snap_pct'] = g['snap_pct'].transform(lambda x: x.shift(1).expanding().mean())

snap_features = snaps[['name_norm', 'team', 'season', 'week',
                        'lag_snap_pct', 'roll3_snap_pct', 'season_snap_pct']]

df = df.merge(snap_features, on=['name_norm', 'team', 'season', 'week'], how='left')
df.drop(columns=['name_norm'], inplace=True)

match_rate = df['lag_snap_pct'].notna().mean()
print(f'Snap count join match rate: {match_rate:.1%} ({df["lag_snap_pct"].notna().sum()} rows matched)')

if match_rate < 0.90:
    print('WARNING: Match rate < 90% — check unmatched players below and add to NAME_OVERRIDES/TEAM_OVERRIDES')
    unmatched = df[df['lag_snap_pct'].isna() & (df['season'] >= 2023) & (df['games_played'] >= 3)]
    if len(unmatched) > 0:
        sample_unmatched = unmatched.groupby('name')['name'].count().sort_values(ascending=False).head(10)
        print(f'\nTop unmatched players:')
        print(sample_unmatched)

Snap count join match rate: 90.1% (8985 rows matched)


In [6]:
# Inspect snap % distribution for matched rows
matched = df[df['lag_snap_pct'].notna()]
print(f'lag_snap_pct percentiles (matched rows):')
for p in [10, 25, 50, 75, 90]:
    print(f'  {p}th: {np.percentile(matched["lag_snap_pct"].dropna(), p):.3f}')

# Check unmatched players (sample)
unmatched = df[df['lag_snap_pct'].isna() & (df['season'] >= 2023) & (df['games_played'] >= 3)]
if len(unmatched) > 0:
    sample_unmatched = unmatched.groupby('name')['name'].count().sort_values(ascending=False).head(10)
    print(f'\nTop unmatched players (may need fuzzy name fix):')
    print(sample_unmatched)

lag_snap_pct percentiles (matched rows):
  10th: 0.210
  25th: 0.400
  50th: 0.650
  75th: 0.830
  90th: 0.930

Top unmatched players (may need fuzzy name fix):
name
Gabe Davis             22
Drew Ogletree          16
Harold Fannin Jr.      13
Tre' Harris            13
Oronde Gadsden         12
Luther Burden III      12
Donald Parham Jr.      10
Scotty Miller           9
Dont'e Thornton Jr.     8
Mecole Hardman Jr.      7
Name: name, dtype: int64


## Feature C — Red Zone Targets

From play-by-play data via `nfl_data_py`. Filters pass plays with `yardline_100 <= 20`.

**New columns:** `cum_rz_targets`, `cum_rz_tds`, `roll3_rz_targets`, `rz_target_share`  
(feature_prep.py will compute `rz_td_rate_eb` from `cum_rz_tds`/`cum_rz_targets` using EB shrinkage)

In [7]:
# PBP download is large (~300 MB per year). nfl_data_py caches locally after first download.
# Note: do NOT use the 'columns' parameter — it breaks parquet loading for some years.
print('Fetching PBP data 2022-2025 (may take a few minutes on first run)...')
pbp_raw = nfl.import_pbp_data([2022, 2023, 2024, 2025])
print(f'PBP rows: {len(pbp_raw):,}  columns: {pbp_raw.shape[1]}')

# Keep only what we need to save memory
keep_cols = [
    'receiver_player_id', 'receiver_player_name',
    'posteam', 'season', 'week',
    'play_type', 'yardline_100', 'touchdown',
]
pbp = pbp_raw[keep_cols].copy()
del pbp_raw

# Filter to red zone pass plays with a receiver
rz = pbp[
    (pbp['play_type'] == 'pass') &
    (pbp['yardline_100'] <= 20) &
    (pbp['receiver_player_id'].notna())
].copy()
print(f'Red zone pass plays with receiver: {len(rz):,}')

# Aggregate per player per game
rz_agg = (
    rz
    .groupby(['receiver_player_name', 'posteam', 'season', 'week'])
    .agg(
        rz_targets_game=('receiver_player_id', 'count'),
        rz_tds_game=('touchdown', 'sum')
    )
    .reset_index()
    .rename(columns={'receiver_player_name': 'name_short', 'posteam': 'team'})
)
rz_agg['season'] = rz_agg['season'].astype(int)
rz_agg['week']   = rz_agg['week'].astype(int)
print(f'\nRZ aggregated rows: {len(rz_agg)}')
print(rz_agg.head(3))

Fetching PBP data 2022-2025 (may take a few minutes on first run)...


2022 done.


2023 done.


2024 done.


2025 done.
Downcasting floats.
PBP rows: 197,362  columns: 397
Red zone pass plays with receiver: 9,759

RZ aggregated rows: 6792
   name_short team  season  week  rz_targets_game  rz_tds_game
0  A.Abdullah  IND    2025     7                1       0.0000
1  A.Abdullah   LV    2022     8                1       0.0000
2  A.Abdullah   LV    2022     9                1       0.0000


In [8]:
# nfl_data_py uses short-format names (e.g. 'J.Chase'). Try joining via nfl.import_ids().
# Fall back to normalized name matching if ID crosswalk is incomplete.

print('Fetching player ID crosswalk...')
ids = nfl.import_ids()
print(f'ID table cols: {list(ids.columns)}')

# Build name_norm → game_logs name mapping from ids table if possible
# Use short_name (nflverse format) → name (full name) crosswalk
# Fall back: normalize both sides to lowercase stripped

rz_agg['name_norm'] = rz_agg['name_short'].str.lower().str.strip()
df['name_norm']     = df['name'].str.lower().str.strip()

# Some names in PBP are 'F.Last' format — try to match against game_logs full names
# Build a short-name → full-name lookup from game_logs (take most recent occurrence)
name_lookup = (
    df[['name_norm', 'name']]
    .drop_duplicates('name_norm', keep='last')
    .set_index('name_norm')['name']
    .to_dict()
)

# Also try matching PBP short name against first-initial.last format of full names
def to_short(full_name):
    """Convert 'First Last' to 'F.Last' for cross-format matching."""
    parts = full_name.strip().split()
    if len(parts) >= 2:
        return f'{parts[0][0]}.{" ".join(parts[1:])}'
    return full_name

# Build reverse lookup: 'F.Last' → 'First Last' from game_logs
short_to_full = {
    to_short(name).lower(): name
    for name in df['name'].unique()
}

def resolve_name(short_lower):
    """Try exact match, then short-name match."""
    if short_lower in name_lookup:
        return name_lookup[short_lower]
    if short_lower in short_to_full:
        return short_to_full[short_lower]
    return None

rz_agg['name_resolved'] = rz_agg['name_norm'].apply(resolve_name)
rz_agg['name_resolved'] = rz_agg['name_resolved'].fillna(rz_agg['name_norm'])  # fallback to short name
rz_agg['name_join'] = rz_agg['name_resolved'].str.lower().str.strip()

print(f'Name resolution: {rz_agg["name_resolved"].notna().mean():.1%} resolved')

Fetching player ID crosswalk...


ID table cols: ['db_season', 'pfr_id', 'yahoo_id', 'height', 'stats_id', 'rotoworld_id', 'draft_pick', 'birthdate', 'draft_year', 'gsis_id', 'draft_ovr', 'pff_id', 'swish_id', 'merge_name', 'name', 'weight', 'team', 'nfl_id', 'college', 'rotowire_id', 'fantasy_data_id', 'sleeper_id', 'cfbref_id', 'espn_id', 'stats_global_id', 'fleaflicker_id', 'draft_round', 'ktc_id', 'cbs_id', 'sportradar_id', 'mfl_id', 'fantasypros_id', 'twitter_username', 'age', 'position']
Name resolution: 100.0% resolved


In [9]:
# Engineer rolling/cumulative RZ features within rz_agg before joining
rz_agg = rz_agg.sort_values(['name_join', 'team', 'season', 'week']).reset_index(drop=True)
g_rz = rz_agg.groupby(['name_join', 'team', 'season'])

# Cumulative RZ targets and TDs (for EB shrinkage in feature_prep.py)
rz_agg['cum_rz_targets'] = g_rz['rz_targets_game'].transform(lambda x: x.shift(1).expanding().sum().fillna(0))
rz_agg['cum_rz_tds']     = g_rz['rz_tds_game'].transform(lambda x: x.shift(1).expanding().sum().fillna(0))

# Rolling 3-game RZ targets (lag-shifted)
rz_agg['roll3_rz_targets'] = g_rz['rz_targets_game'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).sum()
)

# rz_target_share denominator: cumulative team RZ targets from raw game-level totals
# (NOT the sum of per-player cumulatives, which inflates the denominator)
team_game_rz = (
    rz_agg.groupby(['team', 'season', 'week'])['rz_targets_game']
    .sum()
    .reset_index()
    .sort_values(['team', 'season', 'week'])
)
team_game_rz['team_cum_rz_targets'] = (
    team_game_rz.groupby(['team', 'season'])['rz_targets_game']
    .transform(lambda x: x.shift(1).expanding().sum().fillna(0))
)
rz_agg = rz_agg.merge(
    team_game_rz[['team', 'season', 'week', 'team_cum_rz_targets']],
    on=['team', 'season', 'week'],
    how='left'
)
rz_agg['rz_target_share'] = (
    rz_agg['cum_rz_targets'] / rz_agg['team_cum_rz_targets'].replace(0, np.nan)
)

rz_join = rz_agg[['name_join', 'team', 'season', 'week',
                   'cum_rz_targets', 'cum_rz_tds',
                   'roll3_rz_targets', 'rz_target_share']]

# Merge into main df
df = df.merge(
    rz_join,
    left_on=['name_norm', 'team', 'season', 'week'],
    right_on=['name_join', 'team', 'season', 'week'],
    how='left'
).drop(columns=['name_join'], errors='ignore')

# Players with 0 RZ targets get 0 (not NaN)
for col in ['cum_rz_targets', 'cum_rz_tds', 'roll3_rz_targets', 'rz_target_share']:
    df[col] = df[col].fillna(0.0)

# Clean up temp col
df.drop(columns=['name_norm'], inplace=True, errors='ignore')

rz_match = (df['roll3_rz_targets'] > 0).mean()
print(f'Rows with any RZ target history: {rz_match:.1%}')
print(df[['roll3_rz_targets', 'rz_target_share', 'cum_rz_targets', 'cum_rz_tds']].describe())

Rows with any RZ target history: 28.4%
       roll3_rz_targets  rz_target_share  cum_rz_targets  cum_rz_tds
count         9972.0000        9972.0000       9972.0000   9972.0000
mean             1.0879           0.0504          1.8434      0.4617
std              2.0439           0.1012          4.0811      1.1814
min              0.0000           0.0000          0.0000      0.0000
25%              0.0000           0.0000          0.0000      0.0000
50%              0.0000           0.0000          0.0000      0.0000
75%              1.0000           0.0556          1.0000      0.0000
max             15.0000           1.0000         33.0000     10.0000


In [10]:
# Sanity check: top players by rz_target_share in 2024
s24 = df[(df['season'] == 2024) & (df['week'] >= 10)]
top_rz = (
    s24.groupby('name')[['rz_target_share', 'roll3_rz_targets']]
    .mean()
    .sort_values('rz_target_share', ascending=False)
    .head(15)
)
print('Top 15 players by avg rz_target_share (2024, wk10+):')
print(top_rz)

Top 15 players by avg rz_target_share (2024, wk10+):
                   rz_target_share  roll3_rz_targets
name                                                
Drake London                0.3438            4.5000
Trey McBride                0.2731            5.2500
Ja'Marr Chase               0.2650            8.1250
CeeDee Lamb                 0.2648            4.4286
Amon-Ra St. Brown           0.2570            5.8889
George Pickens              0.2393            3.0000
Justin Jefferson            0.2214            5.1111
Wan'Dale Robinson           0.2085            2.5000
Travis Kelce                0.2028            6.0000
Malik Nabers                0.1960            3.1250
Courtland Sutton            0.1955            3.7500
David Njoku                 0.1933            6.8000
Jerry Jeudy                 0.1873            2.7500
Mark Andrews                0.1849            4.0000
Garrett Wilson              0.1846            3.6250


## Save Updated Enriched CSV

In [11]:
# Verify new columns are present
new_cols = [
    'opp_wr_te_td_rate_allowed', 'opp_wr_te_targets_pg_allowed',
    'lag_snap_pct', 'roll3_snap_pct', 'season_snap_pct',
    'cum_rz_targets', 'cum_rz_tds', 'roll3_rz_targets', 'rz_target_share',
]
missing = [c for c in new_cols if c not in df.columns]
if missing:
    print(f'WARNING: Missing columns: {missing}')
else:
    print('All new columns present.')

# Drop the temp opp column before saving
if 'opp' in df.columns:
    df.drop(columns=['opp'], inplace=True)

df.to_csv('data/game_logs_enriched.csv', index=False)
print(f'\nSaved data/game_logs_enriched.csv — {len(df)} rows, {len(df.columns)} columns')
print(f'\nNew column null rates:')
for col in new_cols:
    if col in df.columns:
        print(f'  {col:<40} {df[col].isna().mean():.1%} null')

All new columns present.

Saved data/game_logs_enriched.csv — 9972 rows, 66 columns

New column null rates:
  opp_wr_te_td_rate_allowed                1.1% null
  opp_wr_te_targets_pg_allowed             1.1% null
  lag_snap_pct                             9.9% null
  roll3_snap_pct                           9.9% null
  season_snap_pct                          9.9% null
  cum_rz_targets                           0.0% null
  cum_rz_tds                               0.0% null
  roll3_rz_targets                         0.0% null
  rz_target_share                          0.0% null


In [12]:
# Quick correlation check: how do new features correlate with scored_td?
corr_cols = new_cols + ['scored_td']
available = [c for c in corr_cols if c in df.columns]
corr = df[available].corr()['scored_td'].drop('scored_td').sort_values(ascending=False)
print('Pearson correlation with scored_td (new features):')
print(corr)

Pearson correlation with scored_td (new features):
roll3_rz_targets                0.2908
rz_target_share                 0.2680
cum_rz_targets                  0.2553
cum_rz_tds                      0.2235
season_snap_pct                 0.1942
roll3_snap_pct                  0.1896
lag_snap_pct                    0.1828
opp_wr_te_targets_pg_allowed    0.0018
opp_wr_te_td_rate_allowed      -0.0049
Name: scored_td, dtype: float64


## Next Steps

All three feature groups have been added to `game_logs_enriched.csv`.

`feature_prep.py` auto-detects the new columns and:
- Uses the full `FEATURES` list (25 features) when new columns are present
- Falls back to `FEATURES_V2_BASE` (18 features) if columns are missing
- Applies Beta-Binomial EB shrinkage to `rz_td_rate_eb` (from `cum_rz_tds`/`cum_rz_targets`)
- Fills NaN snap % and RZ features with 0.0

**To retrain the model with the full feature set:**
```bash
cd ml
python train.py      # trains + saves model bundle
python calibrate.py  # re-runs calibration
python evaluate.py   # final holdout metrics
```

Or run notebook `04_model_training.ipynb` end-to-end.